Marketing Mix Modeling (MMM) – Channel ROI Optimization using Classical Econometric Approach

Business Problem
A company wants to understand how Paid Search, Social Media, and Display advertising contribute to revenue and how the next quarter's marketing budget should be allocated to maximize ROI while maintaining the same overall budget.

Business Objective

A company invests in three paid media channels:

Paid Search
Social Media
Display

Management wants to answer:

Which channel contributes the most to revenue?
What is the ROI of each channel?
How should next quarter's budget be allocated?

In [1139]:
# =============================================================================
# STEP 1: IMPORT LIBRARIES
# =============================================================================


#Data Manipulation Libraries
import pandas as pd
import numpy as np

#Visualization library
import plotly.express as px
import plotly.graph_objects as go

#Statistical Modeling Library
import statsmodels.api as sm

#Model Evaluation library
from sklearn.metrics import(
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
from sklearn.model_selection import train_test_split

#Linear Programming
from scipy.optimize import linprog

In [1140]:
# =============================================================================
# STEP 2: GENERATE MARKETING DATASET
# =============================================================================
# Business Assumptions

BASE_REVENUE = 50000

PAID_SEARCH_EFFECT = 3.5
SOCIAL_MEDIA_EFFECT = 2.7
DISPLAY_EFFECT = 1.8

NOISE_STD = 4000
# Set seed for reproducibility
np.random.seed(42)

# Number of observations (2 years of weekly data)
weeks = 104

# Generate weekly marketing spend
df = pd.DataFrame({

    "Week": range(1, weeks + 1),

    "Paid_Search_Spend": np.random.randint(
        12000,
        20001,
        weeks
    ),

    "Social_Media_Spend": np.random.randint(
        8000,
        15001,
        weeks
    ),

    "Display_Spend": np.random.randint(
        5000,
        10001,
        weeks
    )

})

# Generate business noise
noise = np.random.normal(
    loc=0,
    scale= 4000,
    size=weeks
)

# Generate weekly revenue
df["Revenue"] = (

    BASE_REVENUE

    + PAID_SEARCH_EFFECT * df["Paid_Search_Spend"]

    + SOCIAL_MEDIA_EFFECT * df["Social_Media_Spend"]

    + DISPLAY_EFFECT * df["Display_Spend"]

    + noise

).round(2)

# Display first five rows
df.head()

,Week,Paid_Search_Spend,Social_Media_Spend,Display_Spend,Revenue
0,1,19270,8161,8627,156180.59
1,2,19603,12297,6363,160408.39
2,3,12860,9981,6981,141987.60
3,4,17390,8995,6663,149040.23
4,5,17226,14413,6529,156193.09


In [1141]:
# =============================================================================
# STEP 3: EXPLORATORY DATA ANALYSIS
# =============================================================================

#3.1: Datset Overview
#Display first five rows

df.head()

,Week,Paid_Search_Spend,Social_Media_Spend,Display_Spend,Revenue
0,1,19270,8161,8627,156180.59
1,2,19603,12297,6363,160408.39
2,3,12860,9981,6981,141987.60
3,4,17390,8995,6663,149040.23
4,5,17226,14413,6529,156193.09


In [1142]:
#display shape of the dataset

df.shape

(104, 5)

In [1143]:
#data information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Week                104 non-null    int64  
 1   Paid_Search_Spend   104 non-null    int32  
 2   Social_Media_Spend  104 non-null    int32  
 3   Display_Spend       104 non-null    int32  
 4   Revenue             104 non-null    float64
dtypes: float64(1), int32(3), int64(1)
memory usage: 3.0 KB


In [1144]:
#3.2 Missing Value Analysis

df.isnull().sum

<bound method DataFrame.sum of       Week  Paid_Search_Spend  Social_Media_Spend  Display_Spend  Revenue
0    False              False               False          False    False
1    False              False               False          False    False
2    False              False               False          False    False
3    False              False               False          False    False
4    False              False               False          False    False
..     ...                ...                 ...            ...      ...
99   False              False               False          False    False
100  False              False               False          False    False
101  False              False               False          False    False
102  False              False               False          False    False
103  False              False               False          False    False

[104 rows x 5 columns]>

In [1145]:
#3.3 Summary statistics

df.describe().round(2)

,Week,Paid_Search_Spend,Social_Media_Spend,Display_Spend,Revenue
count,104.00,104.00,104.00,104.00,104.00
mean,52.50,16005.41,11729.12,7653.31,151962.05
std,30.17,2368.93,2131.76,1340.10,10517.96
min,1.00,12034.00,8064.00,5146.00,124380.54
25%,26.75,14045.50,9748.25,6659.25,144737.24
50%,52.50,15664.00,11862.00,7794.00,151905.00
75%,78.25,18232.00,13662.00,8702.75,158718.26
max,104.00,19989.00,14938.00,9976.00,178887.18


In [1146]:
#3.4 Revenue Trend

fig = px.line(
    df,
    x= "Week",
    y= "Revenue",
    title= "Weekly Revenue Trend"
)

fig.update_layout(
    xaxis_title= "Week",
    yaxis_title = "Revenue"
)

fig.show()

In [1147]:
#3.5 Marketing Spend Trend

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x = df["Week"],
        y = df["Paid_Search_Spend"],
        name = "Paid Search"
    )
)
fig.add_trace(
    go.Scatter(
        x= df["Week"],
        y = df["Social_Media_Spend"],
        name = "Social Media"
    )
)

fig.add_trace(
    go.Scatter(
        x= df["Week"],
        y = df["Display_Spend"],
        name = "Display"
    )
)

fig.update_layout(
    title = "Weekly Marketing Spend by Channel",
    xaxis_title = "Week",
    yaxis_title = "Spend"
)

fig.show()

In [1148]:
#3.6 Distribution of Marketing Spend

fig = px.histogram(
    df,
    x= "Paid_Search_Spend",
    nbins= 20,
    title = "Distribution of Paid Search Spend"
)

fig.show()

In [1149]:
#Social Media Marketing Spend

fig = px.histogram(
    df,
    x = "Social_Media_Spend",
    nbins= 20,
    title= "Distribution of Social Media Spend"
)

fig.show()

In [1150]:
#Markering spend for Display Ads

fig = px.histogram(
    df,
    x = "Display_Spend",
    nbins= 20,
    title= "Distribution of Display Spend"
)

fig.show()

In [1151]:
#3.7 Correlation Matrix

corr = df.corr(numeric_only= True)

fig = px.imshow(
    corr,
    text_auto= ".2f",
    color_continuous_scale= "Blues",
    title= "Correlation Matrix"
)

fig.show()

In [1152]:
# =============================================================================
# STEP 4: ADSTOCK TRANSFORMATION
# =============================================================================

def adstock(spend,decay):
    adstock_values = []
    previous_effect = 0
    for current_spend in spend:
        current_effect = current_spend + decay * previous_effect
        adstock_values.append(current_effect)
        previous_effect = current_effect
    return adstock_values

In [1153]:
#Apply Adstock
df["Paid_Search_Adstock"]= adstock(
    df["Paid_Search_Spend"],
    decay = 0.5
    )
df["Social_Media_Adstock"] = adstock(
    df["Social_Media_Spend"],
    decay = 0.35
)

df["Display_Adstock"] = adstock(
    df["Display_Spend"],
    decay = 0.20
)

In [1154]:
# Revenue generated from marketing contribution

expected_revenue = (
    BASE_REVENUE
    + PAID_SEARCH_EFFECT * df["Paid_Search_Adstock"]
    + SOCIAL_MEDIA_EFFECT * df["Social_Media_Adstock"]
    + DISPLAY_EFFECT * df["Display_Adstock"]
)

noise = np.random.normal(
    loc=0,
    scale=NOISE_STD,
    size=weeks
)

# Final observed revenue

df["Revenue"] = (

    expected_revenue

    + noise

).round(2)

In [1155]:
#Verify Results
df[[
    "Paid_Search_Spend",
    "Paid_Search_Adstock"
]].head(10)

,Paid_Search_Spend,Paid_Search_Adstock
0,19270,19270.000000
1,19603,29238.000000
2,12860,27479.000000
3,17390,31129.500000
4,17226,32790.750000
5,17191,33586.375000
6,15772,32565.187500
7,15092,31374.593750
8,17734,33421.296875
9,18265,34975.648438


In [1156]:
#Visualisation to compare Original vs Adstock Spend

fig=  go.Figure()

fig.add_trace(
    go.Scatter(
        x= df["Week"],
        y= df["Paid_Search_Spend"],
        name = "Original Spend"
    )
)

fig.add_trace(
    go.Scatter(
        x = df["Week"],
        y= df["Paid_Search_Adstock"],
        name = "Adstocked Spend"
    )
)
fig.update_layout(
    title= "Paid Search: Original vs Adstocked",
    xaxis_title = "Week",
    yaxis_title = "Spend"
)

fig.show()

In [1157]:
# =============================================================================
# STEP 5: MULTIPLE LINEAR REGRESSION
# =============================================================================

#Independent Variables
X = df[[
    "Paid_Search_Adstock",
    "Social_Media_Adstock",
    "Display_Adstock"
]]

#Dependent Variable

y = df["Revenue"]

In [1158]:
# 5.2 Add Constant (Baseline)

X= sm.add_constant(X)

In [1159]:
# 5.3 Train-Test Split the model

split_index = int(len(df)*0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train= y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [1160]:
#5.3 contd.... Fit the model

model = sm.OLS(y_train,X_train).fit()

In [1161]:
#Print Summary

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                Revenue   R-squared:                       0.927
Model:                            OLS   Adj. R-squared:                  0.925
Method:                 Least Squares   F-statistic:                     336.1
Date:                Fri, 12 Jun 2026   Prob (F-statistic):           7.25e-45
Time:                        00:46:10   Log-Likelihood:                -796.81
No. Observations:                  83   AIC:                             1602.
Df Residuals:                      79   BIC:                             1611.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  5.38e+04 

In [1162]:
# =============================================================================
# STEP 6: PREDICT TEST DATA
# =============================================================================

#Generate predictions

y_pred = model.predict(X_test)

In [1163]:
# =============================================================================
# MODEL EVALUATION METRICS
# =============================================================================

r2 =r2_score(y_test,y_pred)

mae= mean_absolute_error(y_test,y_pred)

rmse = np.sqrt(mean_squared_error(y_test,y_pred))

print(f"R2 score: {r2:3f}")
print(f"MAE score:{mae:2f}")
print(f"RMSE score: {rmse:2f}")


R2 score: 0.913688
MAE score:2598.025011
RMSE score: 3257.192776


In [1164]:
# =============================================================================
# ACTUAL VS PREDICTED VISUALISATION
# =============================================================================

comparison_df = pd.DataFrame({
    "Week": df.iloc[split_index:]["Week"],
    "Actual Revenue":y_test,
    "Predicted Revenue": y_pred
})

fig=px.line(
    comparison_df,
    x= "Week",
    y= ["Actual Revenue", "Predicted Revenue"],
    title= "Actual vs Predicted Revenue"
)

fig.show()

In [1165]:

# =============================================================================
# RESIDUAL ANALYSIS
# =============================================================================

comparison_df["Residual"]= (
    comparison_df["Actual Revenue"]-
    comparison_df["Predicted Revenue"]
)

In [1166]:
#Plot Residuals

fig = px.scatter(
    comparison_df,
    x = "Predicted Revenue",
    y= "Residual",
    title = "Residual Plot"
)

fig.add_hline(
    y=0,
    line_dash = "dash"
)

fig.show()

In [1167]:
# =============================================================================
# STEP 7: CHANNEL CONTRIBUTION
# =============================================================================

coefficients = model.params

coefficients

const                   53797.321035
Paid_Search_Adstock         3.368367
Social_Media_Adstock        2.867411
Display_Adstock             1.538020
dtype: float64

In [1168]:
# =============================================================================
# STEP 7.2: CALCULATE WEEKLY CONTRIBUTIONS
# =============================================================================

df["Paid_Search_Contribution"]=(
    coefficients["Paid_Search_Adstock"]
    *
    df["Paid_Search_Adstock"]
)

df["Social_Media_Contribution"]= (
    coefficients["Social_Media_Adstock"]
    *
    df["Social_Media_Adstock"]
)

df["Display_Contribution"]=(
    coefficients["Display_Adstock"]
    *
    df["Display_Adstock"]
)

In [1169]:
# =============================================================================
# STEP 7.3: TOTAL CHANNEL CONTRIBUTION
# =============================================================================

contribution_summary = pd.DataFrame(
    {
        "Channel":["Paid_Search","Social_Media","Display"],
        "Contribution" : [
            df["Paid_Search_Contribution"].sum(),
            df["Social_Media_Contribution"].sum(),
            df["Display_Contribution"].sum()
        ]

    }
)

In [1170]:
contribution_summary["Contribution"] = (
    contribution_summary["Contribution"]
    .round(2)
)

contribution_summary["Contribution_Million"] = (
    contribution_summary["Contribution"] / 1_000_000
).round(2)

In [1171]:
# =============================================================================
# STEP 7.4: CALCULATE CONTRIBUTION PERCENTAGE
# =============================================================================

contribution_summary["Contribution_Percentage"]= (
    contribution_summary["Contribution"]
    / contribution_summary["Contribution"].sum()
    *100
).round(2)

contribution_summary

,Channel,Contribution,Contribution_Million,Contribution_Percentage
0,Paid_Search,11093085.62,11.09,61.73
1,Social_Media,5352268.02,5.35,29.78
2,Display,1526050.37,1.53,8.49


In [1172]:
# =============================================================================
# STEP 7.5: VISUALIZE
# =============================================================================

fig = px.pie(
    contribution_summary,
    names= "Channel",
    values= "Contribution",
    hole = 0.5,
    title= "Marketing Channel Contribution"
)

fig.show()

In [1173]:
# =============================================================================
# STEP 8: ROI FOR EACH CHANNEL
# =============================================================================

roi_summary = pd.DataFrame(
    {
        "Channel": [
            "Paid_Search",
            "Social_Media",
            "Display"
        ],
        "Total_Spend": [
            df["Paid_Search_Spend"].sum(),
            df["Social_Media_Spend"].sum(),
            df["Display_Spend"].sum()
        ],
        "Contribution": [
            df["Paid_Search_Contribution"].sum(),
            df["Social_Media_Contribution"].sum(),
            df["Display_Contribution"].sum()
        ]
    }
)

roi_summary

,Channel,Total_Spend,Contribution
0,Paid_Search,1664563,1.109309e+07
1,Social_Media,1219829,5.352268e+06
2,Display,795944,1.526050e+06


In [1174]:
# =============================================================================
# STEP 8.2: CALCULATE ROI FOR EACH CHANNEL
# =============================================================================

roi_summary["ROI"] = (
    roi_summary["Contribution"]
    / roi_summary["Total_Spend"]
).round(2)

roi_summary = roi_summary.sort_values(
    by= "ROI",
    ascending= False
)
roi_summary

,Channel,Total_Spend,Contribution,ROI
0,Paid_Search,1664563,1.109309e+07,6.66
1,Social_Media,1219829,5.352268e+06,4.39
2,Display,795944,1.526050e+06,1.92


Debugging for low R2

In [1175]:
# =============================================================================
# STEP 8.3: ROI VISUALIZATION
# =============================================================================

fig = px.bar(
    roi_summary,
    x="Channel",
    y= "ROI",
    text= "ROI",
    title= "Marketing Channel ROI"
)
fig.update_traces(
    textposition = "outside"
)
fig.show()

In [1176]:
# =============================================================================
# STEP 9: BUDGET ALLOCATION
# =============================================================================

#Extract coefficients

intercept = model.params["const"]
paid_coef = model.params["Paid_Search_Adstock"]
social_coef = model.params["Social_Media_Adstock"]
display_coef = model.params["Display_Adstock"]


In [1177]:
#Calculate total budget

total_budget = (
    df["Paid_Search_Spend"].sum()
    + df["Social_Media_Spend"].sum()
    + df["Display_Spend"].sum()
)

In [1178]:
#Linear Programming- Objective Function- maximize revenue by minimizing negative revenue

c = [
    -paid_coef,
    -social_coef,
    -display_coef
]

In [1179]:
#Equality Constraint- Entire budget must be allocated

A_eq = [[1,1,1]]
b_eq= [total_budget]

In [1180]:
#Business Constraints

bounds = [
    (0.30*total_budget, 0.60*total_budget),
    (0.20*total_budget,0.50*total_budget),
    (0.10*total_budget,0.30*total_budget)
]

In [1181]:
#Optimize

result = linprog(

    c,

    A_eq=A_eq,

    b_eq=b_eq,

    bounds=bounds,

    method="highs"

)

In [1182]:
#Create Optimization table

optimized_budget = pd.DataFrame({

    "Channel": [

        "Paid Search",

        "Social Media",

        "Display"

    ],

    "Optimized Spend": result.x

})

optimized_budget["Allocation %"] = (
    optimized_budget["Optimized Spend"]

    / total_budget

    * 100

).round(2)

optimized_budget

,Channel,Optimized Spend,Allocation %
0,Paid Search,2208201.6,60.0
1,Social Media,1104100.8,30.0
2,Display,368033.6,10.0


In [1183]:
#Calculate optimized revenue
optimized_revenue = (

    intercept

    + paid_coef * result.x[0]

    + social_coef * result.x[1]

    + display_coef * result.x[2]

)

print(f"Optimized Revenue: ₹{optimized_revenue:,.2f}")

Optimized Revenue: ₹11,223,785.22


In [1184]:
optimization_summary = pd.DataFrame({

    "Channel": [
        "Paid Search",
        "Social Media",
        "Display"
    ],

    "Current Spend": [
        df["Paid_Search_Spend"].sum(),
        df["Social_Media_Spend"].sum(),
        df["Display_Spend"].sum()
    ],

    "Optimized Spend": result.x

})

optimization_summary["Current Allocation %"] = (
    optimization_summary["Current Spend"] / total_budget * 100
).round(2)

optimization_summary["Optimized Allocation %"] = (
    optimization_summary["Optimized Spend"] / total_budget * 100
).round(2)

optimization_summary

,Channel,Current Spend,Optimized Spend,Current Allocation %,Optimized Allocation %
0,Paid Search,1664563,2208201.6,45.23,60.0
1,Social Media,1219829,1104100.8,33.14,30.0
2,Display,795944,368033.6,21.63,10.0


In [1185]:
current_predicted_revenue = (

    intercept

    + paid_coef * df["Paid_Search_Spend"].sum()

    + social_coef * df["Social_Media_Spend"].sum()

    + display_coef * df["Display_Spend"].sum()

)


revenue_improvement = optimized_revenue - current_predicted_revenue

revenue_improvement_percent = (
    revenue_improvement / current_predicted_revenue
) * 100

In [1186]:
optimization_metrics = pd.DataFrame({

    "Metric": [

        "Current Predicted Revenue",

        "Optimized Predicted Revenue",

        "Revenue Improvement",

        "Revenue Improvement %"

    ],

    "Value": [

        round(current_predicted_revenue,2),

        round(optimized_revenue,2),

        round(revenue_improvement,2),

        round(revenue_improvement_percent,2)

    ]

})

optimization_metrics

,Metric,Value
0,Current Predicted Revenue,10382585.73
1,Optimized Predicted Revenue,11223785.22
2,Revenue Improvement,841199.49
3,Revenue Improvement %,8.10


In [1187]:
comparison_df = optimization_summary.melt(

    id_vars="Channel",

    value_vars=[

        "Current Allocation %",

        "Optimized Allocation %"

    ],

    var_name="Scenario",

    value_name="Allocation"

)

fig = px.bar(

    comparison_df,

    x="Channel",

    y="Allocation",

    color="Scenario",

    barmode="group",

    title="Current vs Optimized Budget Allocation"

)

fig.show()

In [1188]:
#correlation

#corr = df[[
    #"Paid_Search_Adstock",
    #"Social_Media_Adstock",
    #"Display_Adstock",
    #"Revenue"
#]].corr()

#corr

In [1189]:
#create correlation color matrix

#fig = px.imshow(
    #corr,
    #text_auto= ".2f",
    #color_continuous_scale= "Blues",
#)

#fig.show()

In [1190]:
#Check for Multicollinearity

#from statsmodels.stats.outliers_influence import variance_inflation_factor

#vif = pd.DataFrame()

#vif["Variable"] = X_train.columns
#vif["VIF"]= [
    #variance_inflation_factor(
       # X_train.values,
       # i
    #)
    #for i in range(X_train.shape[1])
#]
#vif